# Step 2.16: the tick engine on Interactive Brokers' prints

This notebook is the check of step 2.16 (D-67). `just tick-check` ran step 2.14's three strategies on the store's IBKR sessions of MNQZ2026 at fidelity `trades` and at one minute, replayed them through the Python reference engine, and wrote `docs/validation/ibkr-tick-check.md`. The notebook opens what the run left under `<FEFAT_HOME>/workspaces/default/tick-check/`: each session's bar check, the reference engine's fills against the Rust engine's, every difference of the run on prints from the run on bars with its class, the trades of both side by side, and the hashes. It prints counts, dates, times, and tick differences; the side-by-side table holds your data's prices, so its outputs are stripped before a commit, as every notebook's are.

Run `just tick-check` first; without its folder the notebook says so and stops.

In [ ]:
import json
import os

import polars as pl

from fefat.config import resolve_paths

WORK = resolve_paths(os.environ).workspace_dir / "tick-check"
MATCH = "match"
pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_width_chars(140)


def utc(instants):
    return pl.from_epoch(instants, time_unit="ns").dt.replace_time_zone("UTC")


ready = (WORK / "facts.json").is_file()
print(f"tick-check folder ready: {ready}")
facts = json.loads((WORK / "facts.json").read_text(encoding="utf-8")) if ready else {}
strategies = facts["strategies"] if ready else []
if not ready:
    print("Run `just tick-check` first: nothing to open.")

## The sessions and their bars

Each session's prints and the level the run on prints filled it on, the bars rebuilt from its prints against IBKR's own as `fefat data verify-bars` compares them, and IBKR's bars counted against Databento's of the same minutes.

Expected: 2026-09-21 and 2026-09-22 filled on trade prints, their rebuilt bars without a difference; 2026-09-23 filled on 1-minute bars, since the engine cache leaves its prints out, its rebuilt bars apart in the 20:59Z bar alone.

In [ ]:
bars = {row["trade_date"]: row for row in facts["bars"]} if ready else {}
for session in facts["sessions"] if ready else []:
    level = "trade prints" if session["fidelity"] == "trades" else "1-minute bars"
    print(f"{session['trade_date']}: {session['prints']:,} prints, filled on {level}")
    rebuilt = bars[session["trade_date"]]["rebuilt"]
    where = ""
    if rebuilt["differences"]:
        where = f" ({', '.join(rebuilt['columns'])} at {', '.join(rebuilt['minutes'])})"
    compared = f"{rebuilt['compared']:,} compared, {rebuilt['differences']} differences"
    print(f"  rebuilt bars: {compared}{where}")
    counted = bars[session["trade_date"]]["reference"]
    if counted is not None:
        print(
            f"  against {facts['check']['reference_provider']}: {counted['both']:,} minutes in "
            f"both, {counted['volume_differs']} differ in volume, {counted['price_differs']} "
            "in a price"
        )

## The reference engine against the Rust engine

The reference engine replays each strategy from flat on the sessions on prints, and its fills are held to the Rust engine's field by field; the runs on bars are replayed on every session's bars.

Expected: every fill identical, on prints and on bars, for the three strategies.

In [ ]:
for strategy in strategies:
    replayed = strategy["differential"]
    on_prints = f"{replayed['identical']:,} of {replayed['rust_fills']:,}"
    on_bars = f"{replayed['minute_identical']:,} of {replayed['minute_fills']:,}"
    print(f"{strategy['name']}: {on_prints} fills identical on prints, {on_bars} on 1-minute bars")

## Ticks against minutes

Each strategy's trades on prints matched with its trades on bars by the decision that entered them and their side: the trades by outcome, then those that differ, each with its class and the fills on prints less those on bars in ticks. The report documents each class.

Expected: `bar_boundary` the largest class by far, an entry at the open of the bar after the decision against an entry at the first print after the decision's second; a few `bar_path`, `entry_bar_path`, `stop_through`, and `cascade`; no `unknown`.

In [ ]:
for strategy in strategies:
    classified = pl.read_parquet(WORK / f"classified_{strategy['name']}.parquet")
    differing = classified.filter(pl.col("outcome") != MATCH)
    print(
        f"{strategy['name']}: {classified.height} trades on prints and at one minute, "
        f"{differing.height} differing"
    )
    print(classified.group_by("outcome").len("trades").sort("trades", descending=True))
    print(
        differing.select(
            utc(pl.col("entry_decision")).alias("decision"),
            "side",
            "outcome",
            "entry_ticks",
            "exit_ticks",
            "note",
        )
    )

The trades of one strategy side by side, the run on prints' fills beside the run on bars', each fill at its instant and price. Change `SHOWN` to `limit` or `stop` for the others.

In [ ]:
SHOWN = "market"
COLUMNS = ["entry_decision", "side", "entry_ts", "entry_price", "exit_ts", "exit_price"]
side_by_side = None
if ready:
    ticks = pl.read_parquet(WORK / f"trades_{SHOWN}_ticks.parquet")
    minutes = pl.read_parquet(WORK / f"trades_{SHOWN}_minutes.parquet")
    side_by_side = (
        ticks.select(*COLUMNS, "exit_reason")
        .join(
            minutes.select(*COLUMNS, "exit_reason"),
            on=["entry_decision", "side"],
            how="full",
            coalesce=True,
            suffix="_minutes",
        )
        .with_columns(
            utc(pl.col(name)).alias(name)
            for name in (
                "entry_decision",
                "entry_ts",
                "exit_ts",
                "entry_ts_minutes",
                "exit_ts_minutes",
            )
        )
        .sort("entry_decision")
    )
    print(side_by_side)

## Determinism

Each run's fills on a fresh run of the engine and on the runs of its first sessions, by hash.

Expected: every hash equal on the second run and on the prefixes.

In [ ]:
for strategy in strategies:
    hashes = strategy["determinism"]
    prefixes = all(prefix["equal"] for prefix in hashes["prefixes"])
    first, again = hashes["fills_hash"][:16], hashes["second_fills_hash"][:16]
    print(f"{strategy['name']}: fills {first}, again {again}, prefixes equal: {prefixes}")

## Your check

Read `docs/validation/ibkr-tick-check.md`: the bar check should say no differences for 2026-09-21 and 2026-09-22, and the largest fidelity class, `bar_boundary`, should make sense to you: an entry at a bar's open against an entry at the print after the decision. Pick a market `bar_boundary` pair in the tables above and look at its decision's second in the 1-second bars your store builds from the prints, `ff.market.load("MNQZ2026", "1s", <date>, <date>, fidelity="trades", provider="ibkr")`: the run on bars filled at the open of the 1-second bar that opens at the decision, the first print of that second, and the run on prints at the open of the next 1-second bar that holds a print. The last cell prints the verdict.

In [ ]:
if ready:
    identical = all(
        each["differential"]["identical"] == each["differential"]["rust_fills"]
        and each["differential"]["minute_identical"] == each["differential"]["minute_fills"]
        for each in strategies
    )
    unknown = sum(
        each["fidelity"]["unknown"] + each["fidelity"]["restamped"]["unknown"]
        for each in strategies
    )
    clean = sum(row["rebuilt"]["differences"] == 0 for row in facts["bars"])
    print(
        f"Verdict: reference and Rust identical: {identical}; differences without a class: "
        f"{unknown}; rebuilt bars equal the provider's on {clean} of {len(facts['bars'])} "
        "sessions."
    )